In [274]:
import pandas as pd
import numpy as np

In [275]:
df = pd.read_csv('../datasets/titanic.csv')

In [276]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [277]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [278]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [279]:
df.dropna(inplace=True)

In [280]:
df.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64

In [281]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [282]:
cat_cols = X.select_dtypes(exclude='number').columns

In [283]:
cat_cols

Index(['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], dtype='object')

In [284]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

trf1 = ColumnTransformer(
    transformers=[
        ('cat',OneHotEncoder(handle_unknown='ignore'),cat_cols)
    ], remainder='passthrough'
)

pipe = Pipeline([

    ('trf1', trf1),

    ('model', LogisticRegression(max_iter=1000))

])

In [285]:
import warnings
warnings.filterwarnings("ignore")

In [286]:
from sklearn.model_selection import cross_val_score

np.mean(cross_val_score(pipe,X,y,scoring='accuracy',cv=20))

np.float64(0.7533333333333333)

# Feature Construction

In [287]:
X['Cabin'].value_counts()

Cabin
G6             4
B96 B98        4
C23 C25 C27    4
F33            3
D              3
              ..
C91            1
C124           1
C32            1
E34            1
C148           1
Name: count, Length: 133, dtype: int64

In [288]:
X['Deck'] = X['Cabin'].str[0]

In [289]:
X['Deck']

1      C
3      C
6      E
10     G
11     C
      ..
871    D
872    B
879    C
887    B
889    C
Name: Deck, Length: 183, dtype: object

In [290]:
X.columns

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked', 'Deck'],
      dtype='object')

In [291]:
X = X[['Pclass','Sex','Age','Fare','Embarked','Deck']]

In [292]:
new_cat = ['Sex', 'Embarked', 'Deck']

In [293]:
trf2 = ColumnTransformer([
    ('pre',OneHotEncoder(handle_unknown='ignore'),new_cat)
], remainder='passthrough')

In [294]:
trf3 = LogisticRegression()

In [295]:
pipe2 = Pipeline([
    ('trf2',trf2),
    ('trf3',trf3)
])

In [ ]:
cross_val_score(pipe2,X,y,scoring='accuracy',cv=20,error_score='raise').mean()
#Improvement in the accuracy

np.float64(0.7805555555555557)